# 02 — Évaluation de la qualité des données

Identifier les anomalies qui peuvent compromettre l'analyse et préparer les décisions de traitement.
Le notebook 01 décrit les sources ; ici, nous testons des règles explicites et examinons les lignes concernées.

**Kernel : Python (Telco Churn).** Les règles sont dans `configs/validation.yaml`,
les chemins et les cibles dans `configs/base.yaml` et les surcharges d'environnement.
Les données brutes restent intactes : aucune imputation, normalisation ou suppression.


## 1. Chargement et contrat de qualité

`ValidationConfig` valide les règles avec Pydantic ; `DataValidator` les applique.
Les noms des cibles sont repris de `Settings` : ils ne sont pas dupliqués dans le contrat.
Les colonnes supplémentaires sont acceptées, les colonnes attendues sont contrôlées.

Les catégories canoniques et les bornes métier sont des hypothèses documentées dans le YAML.
Un `warning` demande une investigation ; une `error` viole le contrat technique proposé.
Le notebook continue pour rendre toutes les anomalies visibles.


In [ ]:
import pandas as pd
from IPython.display import display

from telco_churn.data.loaders import TelcoDataLoader
from telco_churn.data.schema import ValidationConfig
from telco_churn.data.validation import DataValidator
from telco_churn.logging import configure_logging
from telco_churn.settings import Settings

settings = Settings.load()
configure_logging(settings)
config = ValidationConfig.load(settings)
loader = TelcoDataLoader(settings)

datasets = {"churn": loader.load_raw("churn"), "pilot": loader.load_raw("pilot")}
targets = {"churn": settings.churn_target_column, "pilot": settings.pilot_target_column}
validators = {
    name: DataValidator(config.datasets[name], targets[name], config.example_limit)
    for name in datasets
}
results = {name: validators[name].assess(frame) for name, frame in datasets.items()}

print(f"Environnement : {settings.environment}")
print(f"Règles : {settings.validation_config_path.name}")
display(
    pd.DataFrame(
        {
            name: {
                "rows": len(frame),
                "columns": len(frame.columns),
                "target": targets[name],
                "unique_key": ", ".join(config.datasets[name].unique_key),
            }
            for name, frame in datasets.items()
        }
    ).T
)

### Lire les résultats

- `PASS` : aucune violation parmi les éléments évalués.
- `FAIL` : au moins une violation observée.
- `SKIP` : colonne absente ou aucune observation éligible ; ce n'est pas une réussite.

`n_checked` est le dénominateur du taux : lignes non manquantes pour les domaines,
lignes aux dates valides pour la chronologie, lignes à clé complète pour l'unicité.
Les règles de structure comptent les colonnes et la présence d'observations.
Les règles peuvent se recouvrir : **ne pas additionner les effectifs de lignes en échec**.


## 2. Structure, identifiants et doublons

Le principal est supposé avoir une ligne par `(customer_id, snapshot_date)` ; plusieurs
photographies du même client peuvent être légitimes. L'unicité de `customer_id` dans le
pilote reste une hypothèse à confirmer, signalée comme avertissement.

`duplicate_rows` compte les copies après la première occurrence.
`unique_key` compte toutes les lignes participant à une clé répétée ; les effectifs diffèrent donc.
Les clés incomplètes sont exclues de ce contrôle et signalées par `missing_values`.


In [ ]:
overview_columns = [
    "rule",
    "column",
    "severity",
    "status",
    "n_checked",
    "n_failed",
    "failed_percent",
]
for name, checks in results.items():
    print(name.upper())
    mask = checks["rule"].isin(
        ["required_columns", "non_empty_dataset", "duplicate_rows", "unique_key"]
    )
    mask |= checks["rule"].eq("missing_values") & checks["column"].isin(
        [config.datasets[name].id_column, targets[name], *config.datasets[name].unique_key]
    )
    display(checks.loc[mask, overview_columns])

## 3. Valeurs manquantes : où se concentrent-elles ?

Le contrôle inclut aussi les chaînes vides ou composées d'espaces, au-delà des valeurs
manquantes reconnues par pandas. Les tableaux segmentés affichent les effectifs et les taux ;
une différence entre segments ne prouve pas le mécanisme à l'origine des manquants.
Les petites populations doivent être interprétées avec prudence. Les groupes restent bruts :
`mensuel` et `MENSUEL` ne sont pas fusionnés.


In [ ]:
for name, checks in results.items():
    print(name.upper())
    display(
        checks.loc[
            checks["rule"].eq("missing_values") & checks["status"].ne("PASS"),
            overview_columns,
        ]
    )

missingness_tables = {}
for name, validator in validators.items():
    for group_column in config.datasets[name].group_columns:
        if group_column not in datasets[name]:
            print(f"{name} / {group_column} : regroupement impossible, colonne absente")
            continue
        table = validator.missingness_by_group(datasets[name], group_column)
        missingness_tables[(name, group_column)] = table
        if table["n_missing"].gt(0).any():
            print(f"{name.upper()} — par {group_column}")
            display(table.sort_values(["column", "missing_percent"], ascending=[True, False]))

## 4. Types et cohérence métier

Les nombres doivent être interprétables, finis et dans le domaine déclaré. Les compteurs,
les indicateurs binaires et les cibles doivent être entiers. Les nombres stockés en chaînes
sont signalés séparément, sans convertir la source.

Les catégories sont comparées exactement aux modalités proposées dans le YAML.
Le contrôle de format repère aussi les écarts de casse et d'espaces. La borne d'âge 18–120,
les montants non négatifs et l'interprétation de `nps` comme note individuelle 0–10 sont
à confirmer : ils génèrent des avertissements. Aucun plafond arbitraire n'est imposé à `data_gb`.


In [ ]:
for name, checks in results.items():
    print(name.upper())
    mask = checks["rule"].isin(["numeric_type", "numeric_domain", "category", "category_format"])
    issues = checks.loc[mask & checks["status"].ne("PASS")]
    if issues.empty:
        print("Aucune anomalie détectée par les règles de domaine configurées.")
    else:
        display(issues[overview_columns + ["examples", "action"]])

## 5. Cohérence et couverture temporelles

Vérifier la lecture des dates et `signup_date ≤ snapshot_date` sur les paires valides.
Les valeurs non interprétables sont rapportées séparément ; elles ne sont pas transformées dans la source.
Les bornes ci-dessous décrivent la couverture observée, sans présumer la fréquence attendue.
Le pilote ne contient pas de date configurée : son nom de fichier ne suffit pas à valider sa période.


In [ ]:
for name, checks in results.items():
    print(name.upper())
    if not config.datasets[name].dates:
        print("Pas de date disponible pour vérifier la couverture temporelle.")
        continue
    display(validators[name].temporal_coverage(datasets[name]))
    display(
        checks.loc[
            checks["rule"].isin(["date_parse", "date_order"]), overview_columns + ["examples"]
        ]
    )

## 6. Synthèse des anomalies et actions proposées

Les exemples contiennent la **position de ligne en base zéro**, utilisable avec `df.iloc`,
et quelques valeurs brutes. Ils ne représentent pas toutes les anomalies.
Les erreurs et avertissements sont des constats, pas une décision automatique de nettoyage.


In [ ]:
all_checks = pd.concat(
    [table.assign(dataset=name) for name, table in results.items()], ignore_index=True
)
display(all_checks.groupby(["dataset", "severity", "status"]).size().rename("n_checks").to_frame())

anomalies = all_checks.loc[all_checks["status"].eq("FAIL")].copy()
display(anomalies[["dataset", *overview_columns, "examples", "action"]])

skipped = all_checks.loc[all_checks["status"].eq("SKIP")]
if not skipped.empty:
    print("Contrôles non évalués — ne pas les considérer comme validés")
    display(skipped[["dataset", "rule", "column", "action"]])

### Export optionnel

Passer `export_reports` à `True` pour enregistrer la synthèse complète dans
`artifacts/metrics/data_quality_checks.csv`. Les exemples de lignes restent dans le notebook.


In [ ]:
export_reports = False
if export_reports:
    output_path = settings.project_root / "artifacts/metrics/data_quality_checks.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    all_checks.drop(columns="examples").to_csv(output_path, index=False)
    print(f"Rapport enregistré : {output_path}")

## 7. Décisions à documenter avant le preprocessing

À compléter à partir des résultats, après validation métier :

- Confirmer l'unité d'observation et le traitement des doublons exacts et des clés répétées.
- Expliquer les âges improbables et les frais mensuels négatifs avant de proposer une correction.
- Confirmer les modalités canoniques et les règles de normalisation des catégories.
- Définir le traitement des manquants, notamment de `nps`, sans imputation globale prématurée.
- Confirmer les définitions des cibles, leurs horizons et leur disponibilité temporelle.

Le succès d'un contrôle signifie uniquement que les données respectent la règle configurée.
Ce notebook ne démontre ni l'absence de fuite de cible ni la validité causale du pilote.
Les stratégies de correction et de préparation seront développées dans les étapes suivantes.
